# LocaleOS: know which translations to check

**Veritas AI Hackathon 2026 · AI + Languages track**

LocaleOS translates a small business's English text into Spanish, Hindi, Mandarin, Arabic and Kurmanji, and flags the translations a human should review. This notebook produces every result in our write-up.

**Before you start:** Runtime → Change runtime type → **T4 GPU**. Run the cells in order. Each step saves its work, so if Colab disconnects, run the same cell again and it continues.

*This project is a prototype. Translations of medical, legal or safety text should always be checked by a qualified person.*

## 1. Setup

In [ ]:
!git clone https://github.com/auctacius/utkarshpearl.git
%cd "utkarshpearl/Alans Claude/demo/localeos"
!pip install -q -r requirements.txt

FLORES+ needs a free Hugging Face account. First accept the terms at https://huggingface.co/datasets/openlanguagedata/flores_plus, then log in here.

In [ ]:
from huggingface_hub import login
login()

## 2. Translate the test material (about 20 minutes)
For 400 sentences in each of two FLORES+ splits, this translates English into our five languages with NLLB-200, translates each result back, and compares it with the professional translation.

In [ ]:
!python build_dataset.py --n 400

## 3. Train the flagger and test it on data it never saw
Trains on the `dev` split, tests on `devtest`. Compares against three simpler methods, and tests each language after training only on the other four.

In [ ]:
!python evaluate_qe.py

## 4. Our own data: 60 business strings
Shop, app, clinic, school and notice text we wrote ourselves, to see whether the flagger holds up outside FLORES+.

In [ ]:
!python build_dataset.py --business
!python ratings.py make

**Native-speaker check.** Download the five `data/ratings_<language>.csv` files (Files panel on the left). Each teammate fills in the `rating` column for their own language with `ok` or `bad`, without seeing the flagger's score. Upload the filled files back to `data/` under the same names, then run the next cell.

In [ ]:
!python ratings.py score

## 5. Look at the results

In [ ]:
import json
r = json.load(open('data/results.json'))
print(json.dumps({k: r[k] for k in ('overall', 'languages', 'unseen_language')}, indent=2))
print(json.dumps(r.get('human', 'no native-speaker ratings yet'), indent=2))

## 6. The app
Prints a link that opens LocaleOS in a new tab. It works for whoever runs this notebook, which is what you need to record the demo.

In [ ]:
import subprocess, time, urllib.request
server = subprocess.Popen(['python', 'server.py', '--port', '8000'])
for _ in range(120):
    try:
        urllib.request.urlopen('http://127.0.0.1:8000/api/status'); break
    except Exception:
        time.sleep(1)
from google.colab.output import eval_js
print('Open LocaleOS:', eval_js('google.colab.kernel.proxyPort(8000)'))

## 7. Save the results to GitHub
Download `data/model.json`, `data/results.json` and the filled ratings files, and commit them to the repository so the app's Evidence page shows them for anyone.